In [1]:
from datasets import load_dataset


In [2]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
from transformers import (
    ViTForImageClassification,
    AutoImageProcessor,
    Trainer,
    TrainingArguments
)
from sklearn.model_selection import train_test_split


In [3]:
import numpy as np


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [5]:
dataset = load_dataset("Santhosh2312/real-bogus-test_30k")

print(dataset)
print(dataset["train"][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/394 [00:00<?, ?B/s]

train-00000-of-00012.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

train-00001-of-00012.parquet:   0%|          | 0.00/87.6M [00:00<?, ?B/s]

train-00002-of-00012.parquet:   0%|          | 0.00/344M [00:00<?, ?B/s]

...

train-00009-of-00012.parquet:   0%|          | 0.00/94.0M [00:00<?, ?B/s]

train-00010-of-00012.parquet:   0%|          | 0.00/138M [00:00<?, ?B/s]

train-00011-of-00012.parquet:   0%|          | 0.00/541M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29067 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 29067
    })
})
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=950x362 at 0x78EA3A5C3FE0>, 'label': 0}


In [6]:
model_name = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [23]:
from PIL import Image

def transform(example):

    images = example["image"]
    labels = example["label"]

    processed_images = []

    for img in images:

        # Split stitched image FIRST (while still PIL)
        width, height = img.size
        split_width = width // 3

        panel1 = img.crop((0, 0, split_width, height)).convert("L")
        panel2 = img.crop((split_width, 0, 2*split_width, height)).convert("L")
        panel3 = img.crop((2*split_width, 0, 3*split_width, height)).convert("L") # Fix: Ensure panel3 has the same width as panel1 and panel2

        # Merge the three grayscale panels into a single RGB image
        # panel1 becomes Red channel, panel2 Green, panel3 Blue
        combined_pil_image = Image.merge("RGB", (panel1, panel2, panel3))

        # Process the combined RGB PIL image
        # The processor will output a (3, H, W) tensor directly
        processed_tensor = processor(images=combined_pil_image, return_tensors="pt")["pixel_values"].squeeze(0)

        processed_images.append(processed_tensor)

    return {
        "pixel_values": torch.stack(processed_images),
        "labels": torch.tensor(labels)
    }

In [16]:
full_dataset = dataset["train"]

labels = full_dataset["label"]

train_indices, test_indices = train_test_split(
    np.arange(len(full_dataset)),
    test_size=200,
    stratify=labels,
    random_state=42
)

train_dataset = full_dataset.select(train_indices)
test_dataset = full_dataset.select(test_indices)

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))


Train size: 28867
Test size: 200


In [25]:
labels_train = train_dataset["label"]

train_indices, val_indices = train_test_split(
    np.arange(len(train_dataset)),
    test_size=0.01,
    stratify=labels_train,
    random_state=42
)

final_train_dataset = train_dataset.select(train_indices)
val_dataset = train_dataset.select(val_indices)

print("Final train size:", len(final_train_dataset))
print("Validation size:", len(val_dataset))


Final train size: 28578
Validation size: 289


In [26]:
final_train_dataset = final_train_dataset.with_transform(transform)
val_dataset = val_dataset.with_transform(transform)
test_dataset = test_dataset.with_transform(transform)

In [19]:
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True,
    id2label={0: "bogus", 1: "real"},
    label2id={"bogus": 0, "real": 1}
)


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


In [27]:
training_args = TrainingArguments(
    output_dir="./results_vit_real_bogus_model",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False   # 🔥 THIS LINE FIXES IT
)


In [28]:
def custom_data_collator(features):
    batch = {}
    batch["pixel_values"] = torch.stack([f["pixel_values"] for f in features])
    batch["labels"] = torch.tensor([f["labels"] for f in features])
    return batch

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_train_dataset,
    eval_dataset=val_dataset,
    data_collator=custom_data_collator
)

In [29]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.035068,0.032693
2,0.017600,0.005655
3,0.012490,0.000086
4,0.000399,0.000005
5,0.000089,0.000006


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=17865, training_loss=0.02271746707538791, metrics={'train_runtime': 6806.5515, 'train_samples_per_second': 20.993, 'train_steps_per_second': 2.625, 'total_flos': 1.107283039602905e+19, 'train_loss': 0.02271746707538791, 'epoch': 5.0})

In [30]:
print(dataset["train"].column_names)


['image', 'label']


In [31]:
trainer.evaluate(test_dataset)


{'eval_loss': 0.010900706984102726,
 'eval_runtime': 4.1741,
 'eval_samples_per_second': 47.915,
 'eval_steps_per_second': 5.989,
 'epoch': 5.0}

In [32]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Get predictions
predictions = trainer.predict(test_dataset)

logits = predictions.predictions
labels = predictions.label_ids

# Convert logits to predicted class
preds = np.argmax(logits, axis=-1)

# Compute metrics
accuracy = accuracy_score(labels, preds)
precision = precision_score(labels, preds)
recall = recall_score(labels, preds)
f1 = f1_score(labels, preds)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")


Accuracy : 0.9950
Precision: 0.9907
Recall   : 1.0000
F1 Score : 0.9953


In [33]:
trainer.save_model("vit_real_bogus_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [34]:
processor.save_pretrained("vit_real_bogus_model")


['vit_real_bogus_model/preprocessor_config.json']

In [35]:
!zip -r vit_real_bogus_model.zip vit_real_bogus_model


  adding: vit_real_bogus_model/ (stored 0%)
  adding: vit_real_bogus_model/config.json (deflated 50%)
  adding: vit_real_bogus_model/preprocessor_config.json (deflated 47%)
  adding: vit_real_bogus_model/training_args.bin (deflated 53%)
  adding: vit_real_bogus_model/model.safetensors (deflated 7%)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [36]:
from google.colab import files
files.download('vit_real_bogus_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>